# 📗 자연어 처리 — BERTopic 토픽 모델링

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

앞서 우리는 문서를 **임베딩 → UMAP → 군집화**로 묶었습니다. 하지만 "이 군집는 무슨 주제인가?"는 사람이 직접 봐야 했죠. 이번 시간엔 각 군집의 **대표 키워드를 자동으로 뽑아** 주제를 붙여 주는 **BERTopic** 을 배웁니다. BERTopic 은 지난 시간의 군집 파이프라인에 **c-TF-IDF 키워드 추출**을 더한 도구입니다. 뉴스 1800건을 넣으면 "야구·축구", "금리·환율" 같은 **주제 묶음**이 저절로 나옵니다. 한국어 키워드를 깔끔하게 뽑는 법, 토픽 개수를 조절하는 법, 결과를 시각화하는 법까지 익힙니다.

## ⏪ 복습 — 지난 시간: 임베딩·UMAP·군집화

지난 시간에 만든 파이프라인이 그대로 **BERTopic 의 엔진**이 됩니다.

- **임베딩**: 문장을 768차원 벡터로 (`emb_model.encode`).
- **UMAP**: 고차원 임베딩을 낮은 차원으로 줄여 군집이 잘 되게 (`UMAP`).
- **HDBSCAN**: 밀도로 군집를 묶고, 애매한 문서는 **노이즈(−1)** 로 (`HDBSCAN`).
- 이번 시간엔 여기에 **각 군집의 대표 키워드 뽑기(c-TF-IDF)** 를 더합니다.

**오늘의 목표**

- [ ] **BERTopic** 의 파이프라인(임베딩→UMAP→HDBSCAN→c-TF-IDF)을 말로 설명한다.
- [ ] `fit_transform` 으로 토픽을 만들고 **`get_topic_info`** 로 토픽 표를 읽는다.
- [ ] **`get_topic`·`get_representative_docs`** 로 토픽의 키워드와 대표 문서를 확인한다.
- [ ] **한국어 토크나이저**로 c-TF-IDF 키워드를 깔끔하게 뽑는다.
- [ ] **`min_topic_size`** 로 토픽 수·노이즈를 조절한다.
- [ ] **`nr_topics`·`reduce_topics`·`merge_topics`** 로 토픽 수를 줄인다.
- [ ] **`visualize_*`** 로 토픽을 시각화한다.
- [ ] **`transform`** 으로 새 문서에 토픽을 배정한다(`fit_transform` 과의 차이를 설명한다).
- [ ] **`representation_model`** 로 키워드 뽑는 방식을 바꿔 보고, 어느 쪽이 나은지 **비교해 고른다**.

아래 두 셀을 먼저 실행해 라이브러리와 한국어 임베딩 모델을 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한국어 토크나이저를 준비합니다.
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from bertopic import BERTopic
from kiwipiepy import Kiwi

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

# 지난 단원에서 배운 형태소 분석 — c-TF-IDF 키워드를 한국어 명사로 뽑는 토크나이저(자바가 필요없는 kiwipiepy)
kiwi = Kiwi()

# 불용어 — 11일차에서 배운 방식 그대로: 공개 일반 목록 + 이 데이터의 도메인 불용어
with open('data/stopwords_ko.json', encoding='utf-8') as f:
    STOPWORDS_GENERAL = set(json.load(f))     # 11일차에서 받아 둔 공개 목록 679개

# 이 데이터에서만 무의미한 고빈도어 — 빈도표를 보고 사람이 고른다(11일차 4절)
STOPWORDS_DOMAIN = {'제품', '구매', '사용', '정말', '진짜', '완전', '그냥', '너무', '정도', '많이'}

# 반대로 일반 목록이 '여기서는 의미 있는 말'까지 지우기도 한다 — 되살릴 단어
# ('아이'·'시간' 은 일반 불용어지만, 이 리뷰에서는 '아이에게 사 준 밴드'처럼 주제를 가른다)
KEEP_WORDS = {'아이', '시간'}
KOREAN_STOPWORDS = (STOPWORDS_GENERAL | STOPWORDS_DOMAIN) - KEEP_WORDS

def korean_tokenizer(text):
    """문서에서 의미있는 명사(2글자 이상)만 골라 돌려줍니다."""
    return [t.form for t in kiwi.tokenize(str(text))
            if t.tag.startswith('NN') and len(t.form) > 1 and t.form not in KOREAN_STOPWORDS]



In [ ]:
# [제공 코드] 지난 단원에서 배운 한국어 임베딩 모델을 불러옵니다(문장→768차원 벡터).
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

## 데이터 살펴보기 — 뉴스 헤드라인 1800건

지난 시간과 같은 **뉴스 헤드라인** 데이터(스포츠·경제·IT과학·세계, 각 450건)를 씁니다. BERTopic 에는 **문서 리스트**(`docs`)를 넣고, 임베딩은 지난 시간처럼 **미리 계산해 둔 파일**을 함께 넘겨 속도를 높입니다.

In [ ]:
news = pd.read_csv('data/news_4cat.csv')
print('뉴스 크기:', news.shape)
display(news.head())

# BERTopic 에 넣을 문서 리스트와, 미리 계산한 임베딩
docs = news['title'].tolist()
emb_path = 'data/news_embeddings.npy'
if os.path.exists(emb_path):
    embeddings = np.load(emb_path)                   # 지난 시간에 저장해 둔 것을 그대로 쓴다
else:
    print('저장된 임베딩이 없어 지금 만듭니다 — 수 분 걸릴 수 있어요')
    embeddings = emb_model.encode(docs, show_progress_bar=False)
    np.save(emb_path, embeddings)
print('문서 수:', len(docs), ' / 임베딩 모양:', embeddings.shape)

---
# 1. BERTopic 이란 — 군집에 키워드를 붙인다

## 왜 필요할까요?
지난 시간의 군집화는 문서를 군집으로 묶었지만, 각 군집가 **무슨 주제**인지는 알려 주지 않았습니다. **BERTopic** 은 군집를 묶은 뒤 각 군집에서 **대표 키워드**를 자동으로 뽑아 "이 군집는 이런 주제"라고 이름표를 달아 줍니다.

### 파이프라인 (4단계)
```text
문서 → [1] 임베딩(문장→벡터) → [2] UMAP(차원 축소) → [3] HDBSCAN(밀도 군집) → [4] c-TF-IDF(키워드)
```
- **[1]~[3]** 은 지난 시간에 배운 그대로입니다. BERTopic 이 내부에서 대신 돌려 줍니다.
- **[4] c-TF-IDF**: 한 토픽에 모인 문서를 **하나의 큰 문서**로 합친 뒤, 그 토픽에서 유난히 자주 나오는 단어를 키워드로 뽑습니다(TF-IDF 를 토픽 단위로 확장한 것 — "class-based TF-IDF").

<img src="images/교안/BERTopic_파이프라인.png" width="940">

> 뉴스 헤드라인 1800건을 이 구성으로 돌리면 **토픽 14개 + 노이즈 406건**이 나옵니다(아래에서 직접 확인합니다).

### 표준 구성
지난 시간의 도구를 그대로 BERTopic 에 끼워 넣습니다. **토픽으로 인정할 최소 문서 수**는 `min_cluster_size`(= 여기서 `min_topic_size`) 로 정합니다.
- `UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric='cosine', random_state=42)`
- `HDBSCAN(min_cluster_size=15, metric='euclidean', prediction_data=True)`
- `CountVectorizer(tokenizer=korean_tokenizer, max_df=0.85)` — 한국어 키워드용(뒤 절에서 자세히)

> **`prediction_data=True` 는 왜 붙이나요?** 나중에 **새 문서를 이 모델에 넣어 토픽을 배정**하려면(7절 `transform`), HDBSCAN 이 학습할 때 쓴 정보를 버리지 않고 남겨 둬야 합니다. 이 옵션이 그 '배정에 쓸 데이터'를 보관해 둡니다. 빠뜨리면 학습은 되지만 나중에 `transform` 에서 에러가 납니다.

> **잠깐 — 지난 시간엔 `n_components=2` 였는데 왜 여기선 `5` 인가요?**
> 목적이 다르기 때문입니다.
> - **지난 시간의 2차원**은 "사람이 눈으로 보려고" 줄인 것입니다. 화면이 평면이라 2차원이어야 그림이 됩니다.
> - **여기의 5차원**은 "기계가 군집하려고" 줄인 것입니다. 눈으로 볼 필요가 없으니 2차원까지 내려갈 이유가 없고, **차원을 덜 줄일수록 정보가 덜 깎입니다.**
>
> 그렇다고 768차원 그대로 두지 않는 이유는, 차원이 너무 높으면 점들이 서로 멀어져 **밀도 기반 군집(HDBSCAN)이 잘 안 통하기** 때문입니다. 5차원은 그 사이의 타협점이라 BERTopic 의 기본값이기도 합니다.
>
> 정리하면 — **보여 줄 때는 2차원, 묶을 때는 5차원**입니다.

### 문법
- **`topic_model.fit_transform(docs, embeddings=embeddings)`** → `(각 문서의 토픽 번호, 확률)`. 미리 만든 임베딩을 넘기면 [1]단계를 건너뛰어 빠릅니다.
- **`get_topic_info()`** → 토픽별 번호·크기·이름 표. **토픽 −1 은 노이즈**(어느 주제에도 안 든 문서).

In [ ]:
# 지난 시간 도구를 그대로 끼워 표준 BERTopic 을 구성하는 함수 (반복해서 쓸 예정)
def make_bertopic(min_topic_size=15):
    """UMAP·HDBSCAN·형태소 토크나이저를 끼운 BERTopic 모델을 만든다.

    Args :
    - min_topic_size : int, default 15
        토픽으로 인정할 최소 문서 수. 키우면 토픽 개수가 줄고 노이즈(−1)가 는다.

    Returns :
    - BERTopic : 학습 전 모델. .fit_transform(docs) 로 사용한다.
    """
    umap_model = UMAP(n_components=5, n_neighbors=15, min_dist=0.0,
                      metric='cosine', random_state=42)
    hdbscan_model = HDBSCAN(min_cluster_size=min_topic_size, metric='euclidean', prediction_data=True)
    vectorizer_model = CountVectorizer(tokenizer=korean_tokenizer, max_df=0.85)
    return BERTopic(embedding_model=emb_model, umap_model=umap_model,
                    hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model, verbose=False)

# 표준 구성(min_topic_size=15)으로 토픽을 만든다
topic_model = make_bertopic(15)
topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

info = topic_model.get_topic_info()
print('만들어진 토픽 수(노이즈 -1 포함):', len(info))
print('노이즈(-1) 문서 수:', int((np.array(topics) == -1).sum()))
display(info.head(10))

### 🖐️ 함께 따라하기 — 토픽 표에서 가장 큰 토픽 찾기

`get_topic_info()` 표에서 **노이즈(−1)를 뺀** 실제 토픽 중 가장 큰(문서가 많은) 토픽의 번호를 찾아봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) info = topic_model.get_topic_info() 를 가져온다
# 2) real = info[info['Topic'] != -1] 로 노이즈를 뺀다
# 3) real 은 이미 Count 내림차순이라 첫 행(real.iloc[0])의 Topic·Count 를 출력한다

### ✅ 바로 확인 퀴즈

**1.** BERTopic 의 파이프라인 4단계를 순서대로 말해 보세요.

<details><summary>정답 보기</summary>

**임베딩 → UMAP(차원 축소) → HDBSCAN(군집) → c-TF-IDF(키워드 추출)** 입니다. 앞 세 단계는 지난 시간에 배운 군집 파이프라인이고, 마지막 c-TF-IDF 로 토픽마다 키워드를 뽑습니다.

</details>

**2.** `get_topic_info()` 에서 토픽 번호 **−1** 은 무엇을 뜻하나요?

<details><summary>정답 보기</summary>

**노이즈** 문서 묶음입니다. HDBSCAN 이 어느 주제에도 확실히 넣지 못한 문서들이 −1 로 모입니다.

</details>

---
# 2. 토픽 해석하기 — 키워드와 대표 문서

## 왜 필요할까요?
토픽 번호(0, 1, 2 …)만으로는 그 토픽이 무슨 내용인지 알 수 없습니다. 각 토픽의 **대표 키워드**와 **대표 문서**를 봐야 "아, 이건 야구 토픽이구나" 하고 사람이 해석할 수 있습니다.

### 문법
- **`get_topic(토픽번호)`** → `(단어, 점수)` 목록. 점수(c-TF-IDF)가 높을수록 그 토픽을 잘 대표하는 단어.
- **`get_representative_docs(토픽번호)`** → 그 토픽을 가장 잘 대표하는 **실제 문서** 몇 개.


<img src="images/교안/c-TF-IDF_원리.png" width="900">

> 토픽 번호는 **크기 순**입니다(−1 다음 0이 가장 큰 토픽). 번호 자체엔 의미가 없고, **키워드로 해석**합니다.

In [ ]:
# 토픽 0 의 키워드 — (단어, c-TF-IDF 점수)
print('[토픽 0 키워드]')
for word, score in topic_model.get_topic(0):
    print(f'  {word:10s} {score:.3f}')

# 토픽 0 의 대표 문서 — 실제 헤드라인으로 주제 확인
print('\n[토픽 0 대표 문서]')
for d in topic_model.get_representative_docs(0):
    print('  -', d)

### 🖐️ 함께 따라하기 — 토픽 1의 키워드 top5

토픽 **1**의 키워드 중 상위 5개 단어만 뽑아 출력해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) topic_model.get_topic(1) 로 (단어, 점수) 목록을 가져온다
# 2) 앞 5개([:5])만 돌면서 단어와 점수(소수 셋째)를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 어떤 토픽의 키워드가 `축구, 손흥민, 대표팀, 경기, 감독` 이라면 이 토픽은 대략 무슨 주제일까요?

<details><summary>정답 보기</summary>

**축구(스포츠)** 관련 토픽입니다. c-TF-IDF 로 뽑힌 상위 키워드를 보면 사람이 주제를 짐작할 수 있습니다.

</details>

**2.** 토픽의 키워드는 `get_topic`, 대표 문서는 무슨 함수로 얻나요?

<details><summary>정답 보기</summary>

**`get_representative_docs(토픽번호)`** 입니다. 그 토픽을 가장 잘 대표하는 실제 문서를 돌려줍니다.

</details>

---
# 3. 한국어 키워드 깔끔하게 — 토크나이저

## 왜 필요할까요?
BERTopic 이 키워드를 뽑을 때 문서를 **단어로 쪼개는(토큰화)** 방식이 중요합니다. 기본 방식은 **띄어쓰기**로 자르기 때문에 한국어에서는 **조사가 붙은 덩어리**("경기를", "경기가")나 한 글자 단어가 섞여 키워드가 지저분해집니다. 그래서 **명사만 골라내는 한국어 토크나이저**(SETUP 의 `korean_tokenizer`)를 `CountVectorizer` 에 끼웁니다.

### 문법
- **`CountVectorizer(tokenizer=korean_tokenizer, max_df=0.85)`** — 문서를 한국어 명사로 쪼개고, **85%가 넘는 문서에 나오는 너무 흔한 말**은 버립니다.
- `max_df=0.85` 는 **거의 모든 문서에 나오는 말**을 걸러 냅니다. 리뷰라면 `착용`·`느낌` 처럼 어느 리뷰에나 있는 말이 그렇습니다 — 남겨 두면 모든 토픽의 키워드가 비슷해져 **주제가 안 갈립니다.**

아래에서 **토크나이저 없이** 만든 모델과 **있는** 모델(우리 표준)의 키워드를 비교해 봅니다.

In [ ]:
# 토크나이저 없이(기본 띄어쓰기) 만든 모델 — 키워드가 지저분

# n_components=5   : 5차원으로 축소 (시각화용 2D 가 아니라 군집화 입력용)
# n_neighbors=15   : 이웃 15개를 보고 구조 파악 — 크면 전체 형태, 작으면 국소 구조
# min_dist=0.0     : 점을 최대한 촘촘히 뭉치게 -> 밀도 기반 군집화에 유리
# metric='cosine'  : 임베딩은 방향(각도)이 의미 -> 코사인 거리
# random_state=42  : 결과 재현 고정
plain_umap = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric='cosine', random_state=42)

# min_cluster_size=15   : 군집으로 인정할 최소 문서 수 — 작으면 노이즈로 남는다
# metric='euclidean'    : UMAP 좌표는 직선거리가 의미를 가짐
# prediction_data=True  : 새 문서 배정(transform)에 필요한 정보를 저장
plain_hdbscan = HDBSCAN(min_cluster_size=15, metric='euclidean',
                        cluster_selection_method='eom', prediction_data=True)
plain_model = BERTopic(embedding_model=emb_model, umap_model=plain_umap,
                       hdbscan_model=plain_hdbscan, verbose=False)   # vectorizer 지정 안 함(기본)
plain_model.fit_transform(docs, embeddings=embeddings)

print('[토크나이저 없음] 토픽 0 키워드:')
print('  ', [w for w, _ in plain_model.get_topic(0)])
print('\n[한국어 토크나이저] 토픽 0 키워드:')
print('  ', [w for w, _ in topic_model.get_topic(0)])
print('\n→ 토크나이저를 쓰면 조사·한 글자 단어가 사라지고 의미있는 명사만 남는다')

### 🖐️ 함께 따라하기 — 토크나이저가 뽑은 명사 확인

`korean_tokenizer` 가 한 문장을 어떤 명사들로 쪼개는지 직접 확인해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) sentence = '국가대표 축구팀이 어제 경기에서 극적으로 승리했다' 를 만든다
# 2) korean_tokenizer(sentence) 를 호출해 결과(명사 리스트)를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 한국어 키워드에서 기본(띄어쓰기) 토큰화가 왜 문제가 되나요?

<details><summary>정답 보기</summary>

조사가 붙은 덩어리("경기를", "경기가")나 한 글자 단어가 서로 다른 단어로 잡혀 키워드가 지저분해집니다. 명사만 골라내는 토크나이저를 쓰면 "경기" 같은 의미 단위로 깔끔하게 뽑힙니다.

</details>

**2.** 한국어 토크나이저를 넣었는데도 토픽 키워드에 `오늘`·`때문`·`경우` 처럼 **주제와 무관한 말**이 자꾸 섞여 나옵니다. 어디를 손봐야 할까요?

<details><summary>정답 보기</summary>

제공 셀의 **`KOREAN_STOPWORDS`** 에 그 말들을 추가합니다. `korean_tokenizer` 는 명사만 골라 주지만, `오늘`·`때문`·`경우` 도 엄연히 명사라 그냥 두면 통과합니다. 어떤 말이 **주제어가 아닌지**는 데이터마다 달라 자동으로 알 수 없고, **키워드를 눈으로 보고 사람이 정해 주는 것**이 실무의 정상적인 절차입니다.

</details>

---
# 4. 토픽 크기 조절 — min_topic_size

## 왜 필요할까요?
`min_topic_size`(= HDBSCAN 의 `min_cluster_size`)는 **토픽으로 인정할 최소 문서 수**입니다. 이 값을 바꾸면 토픽 수와 노이즈가 크게 달라집니다.

| min_topic_size | 토픽 수 | 노이즈 | 성격 |
|---|---|---|---|
| **작게**(10) | 많음 | 적음 | 잘게 쪼갬(세부 주제까지) |
| **크게**(50) | 적음 | 많음 | 크고 굵직한 주제만 |

아래에서 10·30·50 을 비교해 실제로 어떻게 달라지는지 봅니다. (각 설정마다 새로 학습하므로 조금 시간이 걸립니다.)

> **재현성 한 줄**: UMAP 에 `random_state=42` 를 고정해 매번 같은 결과가 나오게 했습니다. 다만 여러 라이브러리가 얽혀 있어 딥러닝 계열의 무작위성을 100% 통제하긴 어려우니, 토픽 수는 약간 흔들릴 수 있습니다.

In [ ]:
# min_topic_size 를 10, 30, 50 으로 바꿔 토픽 수·노이즈 비교
rows = []
for size in [10, 30, 50]:
    m = make_bertopic(size)
    t, _ = m.fit_transform(docs, embeddings=embeddings)
    t = np.array(t)
    n_topics = len(set(t)) - (1 if -1 in t else 0)
    n_noise = int((t == -1).sum())
    rows.append({'min_topic_size': size, '토픽 수': n_topics, '노이즈 수': n_noise})
display(pd.DataFrame(rows))
print('작을수록 토픽이 잘게 많이, 클수록 굵직하게 적게 나뉜다')

### 🖐️ 함께 따라하기 — min_topic_size=40 으로 만들기

`make_bertopic(40)` 으로 모델을 학습해 토픽 수와 노이즈 수를 출력해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) make_bertopic(40) 으로 모델 m40 을 만들고 fit_transform(docs, embeddings=embeddings) 한다
# 2) 결과 토픽 배열을 np.array 로 바꿔 토픽 수(-1 제외)와 노이즈 수를 출력한다

### ✅ 바로 확인 퀴즈

**1.** `min_topic_size=50` 으로 올렸더니 토픽은 4개로 깔끔해졌는데 **노이즈가 전체의 절반**을 넘었습니다. 이 모델을 그대로 써도 될까요?

<details><summary>정답 보기</summary>

**목적에 따라 다릅니다.** "이 코퍼스의 큰 줄기 몇 개"만 보고할 거라면 괜찮습니다 — 남은 4개는 그만큼 단단한 주제입니다. 하지만 "모든 문서를 주제별로 분류"하는 것이 목적이라면 절반을 버린 셈이라 실패입니다. **노이즈 비율은 오류가 아니라 "내 목적에 이 설정이 맞는가"를 재는 눈금**으로 읽으세요.

</details>

**2.** 세부 주제까지 잘게 나누고 싶다면 `min_topic_size` 를 크게 할까요, 작게 할까요?

<details><summary>정답 보기</summary>

**작게** 합니다. 작은 군집도 토픽으로 인정되어 토픽이 많아지고 세부 주제까지 잡힙니다.

</details>

---
# 5. 토픽 수 직접 조절 — 합치기

## 왜 필요할까요?
토픽이 너무 잘게 나뉘면 비슷한 토픽끼리 **합쳐** 보기 좋게 만들고 싶을 때가 있습니다. 세 가지 방법이 있습니다.

| 방법 | 무엇 | 방식 |
|---|---|---|
| **`nr_topics=n`** | 목표 토픽 수 지정 | 비슷한 토픽을 **자동으로** n개까지 병합 |
| **`reduce_topics(docs, nr_topics=n)`** | 학습 후 줄이기 | 이미 만든 모델의 토픽을 n개로 병합 |
| **`merge_topics(docs, [[a, b]])`** | 특정 토픽 수동 병합 | 지정한 토픽들을 **직접** 하나로 |

아래에서 잘게 나뉜 모델(min_topic_size=10)을 만들어 **8개로 줄여** 봅니다.

In [ ]:
# 잘게 나뉜 모델을 만든 뒤, 비슷한 토픽을 자동 병합해 8개로 줄인다
many_model = make_bertopic(10)
many_model.fit_transform(docs, embeddings=embeddings)
before = int((many_model.get_topic_info()['Topic'] != -1).sum())   # 노이즈(-1) 제외

many_model.reduce_topics(docs, nr_topics=8)          # 8개로 자동 병합
after = int((many_model.get_topic_info()['Topic'] != -1).sum())
print('병합 전 토픽 수:', before, '→ 병합 후:', after)

# 같은 일을 모델 만들 때 미리 시키는 방법 — 생성자에 nr_topics 를 준다
auto8 = make_bertopic(10)
auto8.nr_topics = 8                                  # 학습하면서 8개까지 자동 병합
auto8.fit_transform(docs, embeddings=embeddings)
print('nr_topics=8 로 만든 모델의 토픽 수:', int((auto8.get_topic_info()['Topic'] != -1).sum()))
display(many_model.get_topic_info()[['Topic', 'Count', 'Name']].head(10))

### 🖐️ 함께 따라하기 — 토픽 0과 1을 수동으로 합치기

위 `many_model` 에서 토픽 **0**과 **1**을 `merge_topics` 로 직접 하나로 합치고, 남은 토픽 수를 확인해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) many_model.merge_topics(docs, [[0, 1]]) 로 토픽 0과 1을 합친다
# 2) 병합 후 토픽 수(get_topic_info 의 Topic 열에서 -1 이 아닌 행 수)를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 비슷한 토픽을 **자동으로** 목표 개수까지 줄이려면 어떤 방법을 쓰나요?

<details><summary>정답 보기</summary>

**`nr_topics=n`**(모델 생성 시) 또는 **`reduce_topics(docs, nr_topics=n)`**(학습 후)입니다. 비슷한 토픽을 알아서 n개까지 병합합니다.

</details>

**2.** 특정 두 토픽만 콕 집어 하나로 합치려면 무엇을 쓰나요?

<details><summary>정답 보기</summary>

**`merge_topics(docs, [[a, b]])`** 입니다. 지정한 토픽 번호들을 수동으로 하나로 병합합니다.

</details>

---
# 6. 토픽 시각화

## 왜 필요할까요?
표만으로는 토픽 전체 그림이 한눈에 안 들어옵니다. BERTopic 은 인터랙티브 그래프 함수를 여럿 제공합니다. 각 함수는 **그림 객체(fig)** 를 돌려주며, 셀 **마지막 줄에 `fig` 를 홀로** 두면 주피터가 자동으로 그려 줍니다.

### 문법
- **`visualize_barchart(top_n_topics=8)`** — 토픽별 키워드 막대그래프.
- **`visualize_heatmap()`** — 토픽끼리 얼마나 비슷한지 히트맵.
- **`visualize_hierarchy()`** — 토픽이 어떻게 묶이는지 계층도.
- **`visualize_documents(docs, embeddings=embeddings)`** — 문서들을 2차원에 흩뿌린 지도.

> **그림을 보고 무엇을 결정하나요?** 히트맵에서 **유난히 진한 칸**(서로 많이 닮은 두 토픽)과 계층도에서 **가장 먼저 붙는 가지**는, 바로 앞 5절의 `merge_topics` 로 **합칠 후보**입니다. 즉 5절이 '어떻게 합치나'를 가르쳤다면, 6절은 **'무엇을 합칠지 어떻게 고르나'**를 알려 줍니다.

> **띄우는 방법은 그림이 몇 개냐에 달렸습니다.**
> - **한 셀에 그림 하나**면 `fig = topic_model.visualize_...()` 로 받아 **마지막 줄에 `fig`** 만 둡니다(아래 세 셀이 그 방식). 주피터가 마지막 줄의 값을 자동으로 렌더합니다.
> - **한 셀에 그림이 여러 개**면 이 방법이 통하지 않습니다 — 주피터는 **마지막 줄 하나만** 렌더하므로 앞의 그림들이 사라집니다. 이럴 땐 그림마다 **`fig.show()`** 를 부르세요.
>
> `plt.show()` 는 matplotlib 용이라 여기선 쓰지 않습니다 — BERTopic 시각화는 **plotly Figure** 입니다.

In [ ]:
# 토픽별 키워드 막대그래프 — 상위 8개 토픽
fig = topic_model.visualize_barchart(top_n_topics=8)
fig

In [ ]:
# 토픽끼리의 유사도 히트맵
fig = topic_model.visualize_heatmap()
fig

In [ ]:
# 토픽 계층도 — 어떤 토픽끼리 가까운지
fig = topic_model.visualize_hierarchy()
fig

### 🖐️ 함께 따라하기 — 문서 지도 그리기

`visualize_documents` 로 문서들이 토픽별로 어떻게 흩어져 있는지 지도를 그려 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) fig = topic_model.visualize_documents(docs, embeddings=embeddings) 로 그림을 만든다
# 2) 이 셀엔 그림이 하나뿐이므로 셀 마지막 줄에 fig 를 홀로 둔다 (여러 개면 fig.show() 를 쓴다)

### ✅ 바로 확인 퀴즈

**1.** 토픽별 키워드를 막대그래프로 보려면 어떤 함수를 쓰나요?

<details><summary>정답 보기</summary>

**`visualize_barchart(top_n_topics=n)`** 입니다. 상위 n개 토픽의 키워드를 막대그래프로 보여 줍니다.

</details>

**2.** 그림 함수의 결과를 화면에 띄우려면 셀 마지막에 무엇을 두나요?

<details><summary>정답 보기</summary>

그림이 **하나뿐이면** `fig = topic_model.visualize_...()` 로 받은 뒤 **마지막 줄에 `fig`** 만 둡니다(주피터가 자동으로 렌더). 다만 **한 셀에 그림이 여러 개면 마지막 하나만 렌더되므로** 그림마다 **`fig.show()`** 를 불러야 합니다.

</details>

---
# 7. 새 문서에 토픽 붙이기 — `fit_transform` vs `transform`

## 왜 필요할까요?
지금까지는 **가지고 있던 1800건**에서 토픽을 찾았습니다. 그런데 실무에서 진짜 하고 싶은 일은 **내일 새로 들어온 기사**를 "이건 무슨 주제"라고 자동으로 분류하는 것입니다.

지난 시간엔 이걸 **손으로** 했습니다 — 군집마다 임베딩 평균으로 대표 벡터를 만들고, 새 문서와 코사인 유사도를 재서 가장 가까운 군집에 넣었죠. BERTopic 은 그 과정을 **`transform` 한 줄**로 대신해 줍니다.

### 두 메서드의 차이 — 헷갈리면 안 되는 지점
| | 하는 일 | 언제 | 토픽이 |
|---|---|---|---|
| **`fit_transform(docs)`** | 토픽을 **새로 찾는다**(학습) | 처음 한 번 | 새로 만들어짐 |
| **`transform(새문서)`** | 이미 찾은 토픽 중 **배정만 한다**(추론) | 그 뒤로 계속 | 그대로 유지됨 |

> **새 문서가 왔다고 `fit_transform` 을 다시 부르면 안 됩니다.** 토픽이 통째로 새로 학습돼 **번호와 내용이 전부 바뀌고**, 앞서 만든 리포트·집계가 전부 어긋납니다. 추론은 반드시 `transform` 입니다.

### 문법
- **`topic_model.transform(새문서리스트, embeddings=새임베딩)`** → `(배정된 토픽 번호, 확률)`.
- 학습 때와 **같은 임베딩 모델**로 새 문장을 벡터화해 넘겨야 합니다(`emb_model.encode`).
- 어느 토픽에도 충분히 가깝지 않으면 **−1(노이즈)** 로 배정될 수 있습니다.
- 내부에서는 학습 때 만든 UMAP 으로 새 임베딩을 같은 공간에 투영한 뒤 HDBSCAN 이 가장 가까운 토픽을 고릅니다 — **그래서 표준 구성에 `prediction_data=True` 가 필요했습니다**(1절).

In [ ]:
# 새 헤드라인 3개를 이미 학습된 topic_model 에 넣어 토픽을 배정한다 (재학습 아님)
new_titles = [
    '손흥민 멀티골로 팀 역전승',
    '한국은행 기준금리 동결 결정',
    '차세대 인공지능 반도체 양산 시작',
]
new_emb = emb_model.encode(new_titles)          # 학습 때와 같은 임베딩 모델
new_topics, _ = topic_model.transform(new_titles, embeddings=new_emb)

for title, tid in zip(new_titles, new_topics):
    tid = int(tid)
    words = [w for w, _ in topic_model.get_topic(tid)[:5]] if tid != -1 else ['(노이즈)']
    print(f'{title}\n  -> 토픽 {tid} | 키워드: {words}')

print('\n토픽 수는 그대로:', int((topic_model.get_topic_info()['Topic'] != -1).sum()), '개' , '— transform 은 배정만 하고 토픽을 바꾸지 않는다')

### 🖐️ 함께 따라하기 — 엉뚱한 문장은 어디로 갈까

위 세 문장은 뉴스 네 분야에 잘 맞는 문장이었습니다. 이번엔 **이 코퍼스에 없는 주제**의 문장을 넣어 어느 토픽에 배정되는지, 혹시 노이즈(−1)로 빠지는지 확인해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) odd = ['오늘 저녁 된장찌개 끓이는 법'] 을 만든다
# 2) emb_model.encode(odd) 로 임베딩해 topic_model.transform(odd, embeddings=...) 을 부른다
# 3) 배정된 토픽 번호를 출력하고, -1 이 아니면 그 토픽의 키워드 top5 도 함께 출력한다

### ✅ 바로 확인 퀴즈

**1.** 어제 학습해 둔 모델에 오늘 들어온 기사 100건의 토픽을 붙이려 합니다. `fit_transform` 과 `transform` 중 무엇을 쓰고, 다른 하나를 쓰면 무슨 일이 벌어지나요?

<details><summary>정답 보기</summary>

**`transform`** 을 씁니다. `fit_transform` 을 부르면 어제의 토픽이 사라지고 **토픽을 처음부터 다시 학습**하므로, 토픽 번호와 내용이 바뀌어 어제 만든 리포트·집계와 대조가 불가능해집니다.

</details>

**2.** `transform` 이 어떤 새 문서에 **−1** 을 돌려줬습니다. 이건 오류인가요?

<details><summary>정답 보기</summary>

오류가 아닙니다. **어느 토픽에도 충분히 가깝지 않다**는 정상적인 답입니다. 학습 코퍼스에 없던 주제(예: 요리)의 문서가 들어오면 −1 이 나올 수 있고, 이는 "억지로 끼워 넣지 않는다"는 HDBSCAN 의 설계 그대로입니다.

</details>

---
## 🚀 응용 클론코딩 — 뉴스 토픽 리포트

표준 모델(`topic_model`)의 각 토픽에 대해 **키워드·크기·대표 문서**를 정리한 리포트를 만들어 봅니다. 아래 지시에 따라 직접 완성해 보세요.

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) info = topic_model.get_topic_info() 에서 노이즈(-1)를 뺀 real 을 만든다
# 2) real 의 각 행(토픽)에 대해 반복하며:
#    2-1) 토픽 번호 tid 와 문서 수(Count)를 출력한다
#    2-2) get_topic(tid) 의 상위 5개 키워드(단어만)를 출력한다
#    2-3) get_representative_docs(tid) 의 첫 문서 1개를 대표 문서로 출력한다
# 3) 상위 5개 토픽(real.head(5))만 리포트한다

---
# 8. 키워드 뽑는 **방식**을 바꿔 보기 — `representation_model`

## 왜 필요할까요?
3절에서는 **토크나이저**를 바꿔 키워드를 다듬었습니다. 그런데 더 근본적으로 **키워드를 고르는 방식 자체**를 바꿀 수도 있습니다. 지금까지 쓴 c-TF-IDF 는 "그 토픽에서 **유난히 자주** 나오는 단어"를 고릅니다. 다른 방법도 있습니다 — **KeyBERT 방식**은 "그 토픽의 **문서 임베딩과 뜻이 가까운** 단어"를 고릅니다. 빈도가 아니라 **의미로** 고르는 것이죠.

또 하나, 기본 키워드를 보면 `금융·작년·종합·주식` 처럼 **주제와 상관없는 말**(`작년`·`종합`)이 섞이기도 합니다. **MMR**(Maximal Marginal Relevance) 은 후보 중에서 **서로 겹치지 않는 단어**를 골라 이런 군더더기를 밀어냅니다.

BERTopic 은 이 셋을 모두 **`representation_model`** 자리에 끼우게 해 둡니다 — UMAP·HDBSCAN·CountVectorizer 를 끼웠던 것과 똑같은 방식입니다.

| 방식 | 무엇을 기준으로 고르나 | 파라미터 |
|---|---|---|
| **c-TF-IDF** (기본) | 그 토픽에서 **유난히 자주** 나오는 단어 | — |
| **KeyBERTInspired** | 문서 임베딩과 **뜻이 가까운** 단어 | — |
| **MaximalMarginalRelevance** | 후보 중 **서로 안 겹치는** 단어 | `diversity` (0~1) |

\

| diversity | 동작 |
|---|---|
| 0.0 | 다양성 무시, 순수 관련성 순위 그대로 (c-TF-IDF 상위 키워드와 거의 동일) |
| 0.3 (낮음) | 관련성 위주, 약간의 중복 제거만 |
| 0.7 (높음) | 다양성 위주, 의미가 겹치는 키워드는 배제하고 서로 다른 측면의 단어를 강제로 섞음 |
| 1.0 | 관련성 거의 무시, 최대한 서로 다른 단어들만 선택 (품질 저하 위험) |

### 문법
- **`from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance`** — 추가 설치 없이 BERTopic 에 들어 있습니다.
- **`BERTopic(..., representation_model=KeyBERTInspired())`** — 나머지 구성은 그대로 둡니다.
- **`MaximalMarginalRelevance(diversity=0.3)`** — `diversity` 가 클수록 서로 다른 말을 고릅니다.

> 군집화(임베딩→UMAP→HDBSCAN)는 **전혀 바뀌지 않습니다.** 문서가 묶이는 결과는 같고, **그 묶음에 어떤 단어를 붙일지**만 달라집니다.

In [ ]:
# 같은 구성에 representation_model 만 갈아 끼워 키워드 뽑는 방식을 바꾼다
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

reps = {'KeyBERT': KeyBERTInspired(),
        'MMR 0.3': MaximalMarginalRelevance(diversity=0.3),
        'MMR 0.7': MaximalMarginalRelevance(diversity=0.7)}

models = {}
for name, rep in reps.items():
    m = make_bertopic(15)                 # 나머지 구성은 전부 동일
    m.representation_model = rep
    m.fit_transform(docs, embeddings=embeddings)
    models[name] = m

# 기본(c-TF-IDF) 과 나란히 놓고 비교한다
rows = []
for t in range(4):
    row = {'토픽': t, 'c-TF-IDF (기본)': ', '.join(w for w, _ in topic_model.get_topic(t)[:5])}
    for name, m in models.items():
        row[name] = ', '.join(w for w, _ in m.get_topic(t)[:5])
    rows.append(row)
display(pd.DataFrame(rows).set_index('토픽'))

**표를 읽어 봅시다 — 세 방식이 서로 다른 성격을 보입니다.**

| 토픽 | c-TF-IDF (기본) | KeyBERT | MMR 0.3 | MMR 0.7 |
|---|---|---|---|---|
| 0 | 금융·작년·종합·주식 | 주식·손실·증권·위기 | **금융·주식·코로나·분기** | 금융·코로나·분기·투자 |
| 2 | 개발·기술·연구·우주 | 선정·세계·수상·훈장 | **개발·연구·우주·과학** | 우주·교수·전지·생산 |

- **c-TF-IDF**: 무난하지만 `작년`·`종합` 처럼 **주제와 상관없는 말**이 섞입니다. 뉴스에 흔한 단어가 빈도로 밀고 올라온 것입니다.
- **KeyBERT**: 이 데이터에서는 `선정`·`수상`·`훈장` 처럼 **어느 기사에나 어울리는 추상어**가 올라와 주제가 오히려 흐려졌습니다. **더 최신 방식이라고 늘 나은 것이 아닙니다.**
- **MMR 0.3**: 겹치는 말을 밀어내니 `작년`·`종합` 이 빠지고 `코로나`·`분기` 처럼 **내용이 있는 단어**가 들어왔습니다. 이 데이터에선 가장 읽힙니다.
- **MMR 0.7**: 다양성을 너무 올리면 정작 **핵심어까지 밀려납니다** — 토픽 2 에서 `개발`·`연구` 가 빠지고 `전지`·`생산` 만 남아 무슨 주제인지 알기 어려워졌습니다.

> **여기서 배울 것 두 가지.** ① `diversity` 같은 손잡이는 **적당한 값**이 있습니다 — 0.3 은 좋아졌지만 0.7 은 오히려 나빠졌죠(4절의 `min_topic_size` 와 같은 이야기입니다). ② **새 도구가 늘 더 좋지는 않습니다.** 어느 방식이 나은지는 **데이터를 보고 비교해서** 정합니다. 방법을 아는 것보다 **여러 결과를 나란히 놓고 고르는 습관**이 실무의 실력입니다.

### 🖐️ 함께 따라하기 — 군집 결과는 정말 그대로일까

"표현만 바뀌고 군집화는 안 바뀐다"고 했습니다. 정말 그런지 **두 모델의 토픽 수와 문서별 토픽 배정**을 직접 비교해 확인해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 두 모델의 토픽 수(노이즈 -1 제외)를 각각 세어 출력한다
#    (get_topic_info() 의 Topic 열에서 -1 이 아닌 행 수)
# 2) 세 모델 각각에 대해 topic_model.topics_ 와 같은지 == 로 비교해 출력한다

### ✅ 바로 확인 퀴즈

**1.** `representation_model` 을 바꿨더니 토픽 **개수**가 달라졌다면, 무엇을 의심해야 하나요?

<details><summary>정답 보기</summary>

**표현 모델 말고 다른 것도 함께 바뀌었다**고 의심해야 합니다. 표현 모델은 이미 묶인 토픽에 **단어를 붙이는 단계**라 군집 결과에 영향을 주지 않습니다. 개수가 달라졌다면 `min_topic_size` 나 `random_state` 같은 군집화 쪽 설정이 함께 바뀐 것입니다.

</details>

**2.** 상사가 "KeyBERT 가 더 최신이니 그걸 쓰자"고 합니다. 어떻게 답하는 것이 좋을까요?

<details><summary>정답 보기</summary>

**"둘 다 돌려서 키워드를 나란히 보고 정하자"** 가 맞습니다. 실제로 이 뉴스 데이터에서는 기본 c-TF-IDF 가 더 읽히는 키워드를 냈습니다. 어느 방식이 나은지는 **데이터마다 다르고**, 비교는 몇 줄이면 됩니다.

</details>

---
## 이번 강의 정리

| 목적 | 함수 |
|---|---|
| 토픽 만들기 | `topic_model.fit_transform(docs, embeddings=...)` |
| 토픽 표 보기 | `get_topic_info()` |
| 키워드·대표문서 | `get_topic(tid)` / `get_representative_docs(tid)` |
| 한국어 키워드 | `CountVectorizer(tokenizer=korean_tokenizer, max_df=0.85)` |
| 토픽 크기 조절 | `min_topic_size`(HDBSCAN `min_cluster_size`) |
| 토픽 수 줄이기 | `nr_topics` / `reduce_topics` / `merge_topics` |
| 시각화 | `visualize_barchart` / `visualize_heatmap` / `visualize_hierarchy` / `visualize_documents` |
| 새 문서 배정(추론) | `transform(새문서, embeddings=...)` — 재학습 아님 |
| 키워드 뽑는 방식 바꾸기 | `representation_model=` `KeyBERTInspired()` / `MaximalMarginalRelevance(diversity=)` |

- **BERTopic = 임베딩 → UMAP → HDBSCAN → c-TF-IDF**. 지난 시간 군집에 **키워드 추출**을 더한 것.
- **한국어는 토크나이저**가 키워드 품질을 좌우하고, **`max_df=0.85`** 로 어느 문서에나 나오는 흔한 말을 걸러야 토픽끼리 구별됩니다.
- **`min_topic_size`** 로 토픽의 굵기를, **병합**으로 토픽 수를 조절합니다.

---
## 🔮 맛보기 — 토픽에 **이름**을 자동으로 붙이기

여기까지 오면 토픽은 **번호(0, 1, 2 …)와 키워드 목록**까지 나옵니다. 그런데 보고서에 쓰려면 "토픽 3" 이 아니라 **"금융시장 동향"** 같은 **사람이 읽는 이름**이 필요하죠. 지금까지 그 이름은 키워드를 보고 **사람이** 붙였습니다(교안_01 응용에서 직접 해 봤죠).

이 마지막 한 걸음은 **LLM 에게 시킬 수 있습니다** — 키워드와 대표 문서를 주고 "이 토픽에 이름을 붙여 줘" 라고 하면 됩니다. 다음 단원에서 배울 **OpenAI API** 가 바로 그 도구입니다.

게다가 BERTopic 은 이걸 **직접 끼울 수 있게** 만들어 두었습니다. `representation_model` 에 LLM 표현 모델을 넣으면, 학습하면서 토픽마다 알아서 이름을 지어 `get_topic_info()` 의 **`Name` 컬럼**에 담아 줍니다 — 우리가 따로 반복문을 돌릴 필요가 없습니다.

> **이 절은 예고편입니다.** 코드를 지금 이해하지 못해도 괜찮습니다 — API 사용법은 다음 단원에서 제대로 배웁니다. 여기서는 **"토픽 모델링의 마지막 칸을 LLM 이 채운다"** 는 그림만 보면 됩니다.

> **키가 없어도 됩니다.** `.env` 에 `OPENAI_API_KEY` 가 있으면 실제로 호출하고, 없으면 이 셀은 **건너뜁니다**(과금 없음). 키는 `.env.example` 을 `.env` 로 복사해 채우면 됩니다.

In [ ]:
# [맛보기] BERTopic 에 'LLM 표현 모델'을 끼워, 토픽 이름을 자동으로 짓게 한다 (다음 단원 내용)
import os
from dotenv import load_dotenv

load_dotenv('.env')                 # 이 폴더의 .env 에서 키를 읽는다

if not os.getenv('OPENAI_API_KEY'):
    print('OPENAI_API_KEY 가 없어 이 맛보기는 건너뜁니다.')
    print('.env.example 을 .env 로 복사해 키를 채우면 실제로 이름을 받아 볼 수 있어요(다음 단원에서 배웁니다).')
else:
    from openai import OpenAI
    from bertopic.representation import OpenAI as OpenAIRepresentation

    # 토픽마다 '키워드 + 대표 문서'를 넣어 이름을 물어보는 틀
    #  [KEYWORDS]·[DOCUMENTS] 자리에 BERTopic 이 그 토픽의 값을 채워 넣는다
    prompt = ('다음은 한국어 뉴스 헤드라인 토픽입니다.\n'
              '대표 헤드라인:\n[DOCUMENTS]\n'
              '키워드: [KEYWORDS]\n'
              '이 토픽의 이름을 한국어 10자 이내로 하나만 답하세요. 설명 없이 이름만.')

    rep_model = OpenAIRepresentation(client=OpenAI(), model='gpt-4o-mini',
                                     chat=True, nr_docs=4, prompt=prompt)

    # 표준 구성 그대로에 representation_model 만 끼운다 — 나머지는 지금까지와 똑같다
    named_model = make_bertopic(15)
    named_model.representation_model = rep_model
    named_model.fit_transform(docs, embeddings=embeddings)

    named_info = named_model.get_topic_info()
    print('LLM 이 이름을 붙인 토픽 표')
    display(named_info[['Topic', 'Count', 'Name']].head(8))
    print('Name 컬럼이 0_금융_작년_종합 같은 키워드 나열에서 사람이 읽는 이름으로 바뀌었다')

### ✅ 바로 확인 퀴즈

**1.** LLM 에게 토픽 이름을 시킬 때, 키워드만 주지 않고 **대표 문서도 함께** 준 이유는 무엇일까요?

<details><summary>정답 보기</summary>

키워드만으로는 **뜻이 갈리는 토픽**이 있기 때문입니다. 예를 들어 `크림·피부·발림` 만 보면 선크림인지 로션인지 알기 어렵지만, 대표 문서를 함께 읽으면 정체가 분명해집니다. 우리가 2절에서 **키워드와 대표 문서를 함께 봐야 한다**고 배운 것과 같은 이유입니다.

</details>

**2.** 이 맛보기에서 BERTopic 이 한 일과 LLM 이 한 일은 각각 무엇인가요?

<details><summary>정답 보기</summary>

**BERTopic** 은 문서를 묶고(임베딩→UMAP→HDBSCAN) 각 군집의 **키워드를 뽑는** 데까지 했습니다. **LLM** 은 그 키워드와 대표 문서를 읽고 **사람이 읽을 이름을 지어** 주었습니다. 토픽을 *찾는* 것은 여전히 BERTopic 이고, LLM 은 마지막 **해석·명명**만 도왔습니다.

</details>

## ⏭️ 예고 — 다음 단원: OpenAI API 활용

지금까지는 문서를 **묶고 주제 키워드를 뽑는** 데까지 왔습니다. 다음 단원에서는 **OpenAI API** 를 활용하는 방법을 배웁니다. 오늘 익힌 임베딩·토픽 모델링이 그 바탕이 됩니다. 수고하셨습니다!